# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL from [Senscience](https://doi.org/10.71728/senscience.vq4a-28xa).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL from FAIR²
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset Loaded:")
print(f"Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}\nKeywords: {getattr(metadata, 'keywords', None)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the record sets in the dataset, their `@id` fields, and, for each, list the fields and columns (by `@id`) to understand the structure before data extraction.

In [ ]:
print("Available Record Sets (by @id):")
for rs in metadata.record_sets:
    print(f"- RecordSet name: {rs.name} (ID: {rs.id})")
    print("  Fields and Columns:")
    for field in rs.fields:
        print(f"    • Field: {getattr(field, 'name', '')} (ID: {field.id}, DataType: {getattr(field, 'data_type', None)})")
        # Show columns for this field
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        - Column: {getattr(col,'name','')} (ID: {col.id})")
print("\nIf no record sets appear, check that the dataset conforms to the Croissant schema with record sets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s, as listed in the overview above.

**Note:** Replace `<record_set_id>` below with the actual `@id` of a record set you want to explore. For this dataset, there is typically a main record set. We'll use the first one found.

In [ ]:
# Identify record set IDs
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"Record Set IDs found: {record_set_ids}")

# Load all record sets into pandas DataFrames (by @id)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")

# Use the first record set for demonstration
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"\nSample of [{main_record_set_id}] Data:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets present in this dataset.")

## 4. Exploratory Data Analysis (EDA)
We will perform common EDA steps: filtering records, normalizing a numeric field, and grouping data. This example will:
- Select a numeric field (for example: `cr:field/normalized_abundance`)
- Filter for values above a threshold
- Normalize the field (z-score)
- Optionally group by a categorical field (e.g., `cr:field/sample_id`)

**Note:** Replace the field `@id`s below with ones appropriate from your data overview.

In [ ]:
# Set the record set and field @ids based on your overview

record_set_id = main_record_set_id
# Example field IDs (replace with actual ones printed in overview cell)
numeric_field_id = None  # e.g., 'cr:field/normalized_abundance'
group_field_id = None    # e.g., 'cr:field/sample_id'

df = dataframes[record_set_id]

if numeric_field_id is None:
    # Try to infer a suitable numeric field
    # E.g., columns containing 'abundance', 'MW', etc.
    possible = [col for col in df.columns if df[col].dtype in ['float64','int64'] or 'abundance' in col or 'MW' in col]
    print(f"Auto-inferred candidate numeric fields: {possible}")
    if possible:
        numeric_field_id = possible[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric field auto-identified. Please set 'numeric_field_id'.")

if group_field_id is None:
    possible_groups = [col for col in df.columns if 'sample' in col or 'group' in col or 'source' in col]
    if possible_groups:
        group_field_id = possible_groups[0]
        print(f"Using group field: {group_field_id}")

try:
    # Drop rows with missing numeric values
    filtered_df = df.dropna(subset=[numeric_field_id])
    threshold = filtered_df[numeric_field_id].mean()  # Example filter: above the mean
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group if group field exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5):\n{grouped_df.head()}")
except Exception as e:
    print(f"Error in EDA: {e}. Check that field IDs are set and correct.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We show as an example a histogram of the selected numeric field, a boxplot by group (if a group field is present), and a correlation heatmap of numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if numeric_field_id and numeric_field_id in df.columns:
    df[numeric_field_id].dropna().plot(kind='hist', bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field_id and numeric_field_id:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showfliers=False)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

# Plot correlation heatmap for numeric columns
num_cols = df.select_dtypes(include=['float64','int64']).columns
if len(num_cols) >= 2:
    plt.figure(figsize=(10,8))
    sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
    plt.title("Correlation of Numeric Fields")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using standardized Croissant metadata and the `mlcroissant` library. We:
- Inspected record sets and their structure using `@id`
- Loaded data into DataFrames for analysis
- Performed example EDA with field normalization/grouping
- Visualized numeric field distributions and relationships

You can build upon this template to perform advanced analysis, modeling, or to integrate this data with other omics or clinical datasets for further research. Refer to the [mlcroissant documentation](https://mlcommons.org/croissant/) for more advanced data access patterns.